In [1]:
print("FastAPI setup cell ✅")


FastAPI setup cell ✅


In [2]:
import math
from pathlib import Path

import numpy as np
import pandas as pd

from fastapi import FastAPI
from pydantic import BaseModel, Field

from sklearn.metrics.pairwise import cosine_similarity

print("Libraries imported successfully! ✅")


Libraries imported successfully! ✅


In [3]:
DATA_PATH = "../data/processed/manali_travel_enriched.csv"
EMBEDDING_PATH = "../data/embeddings/manali_place_embeddings.npy"

df = pd.read_csv(DATA_PATH)
place_embeddings = np.load(EMBEDDING_PATH)

print("Dataset shape:", df.shape)
print("Embedding shape:", place_embeddings.shape)


Dataset shape: (20, 41)
Embedding shape: (20, 384)


In [4]:
FEATURE_COLUMNS = [
    "nature",
    "history",
    "culture",
    "adventure",
    "photography",
    "shopping",
    "religious",
    "family"
]

X_STRUCTURED = (
    df[FEATURE_COLUMNS]
    .fillna(0)
    .astype(float)
)

print(FEATURE_COLUMNS)


['nature', 'history', 'culture', 'adventure', 'photography', 'shopping', 'religious', 'family']


In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sentence_transformers import SentenceTransformer

df["recommendation_text"] = (
    df["name"].fillna("") + " "
    + df["category"].fillna("") + " "
    + df["travel_tags"].fillna("")
)

tfidf_vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2)
)

tfidf_matrix = tfidf_vectorizer.fit_transform(
    df["recommendation_text"].astype(str)
)

semantic_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Recommendation components ready! ✅")


d:\college_work\PG\linkedIn_projects\TravelMate-AI\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3658.59it/s]


Recommendation components ready! ✅


In [6]:
def minmax(series):
    series = series.astype(float)

    if series.max() == series.min():
        return pd.Series(
            np.ones(len(series)),
            index=series.index
        )

    return (
        (series - series.min())
        / (series.max() - series.min())
    )


def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0

    lat1 = math.radians(lat1)
    lon1 = math.radians(lon1)
    lat2 = math.radians(lat2)
    lon2 = math.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        math.sin(dlat / 2) ** 2
        + math.cos(lat1)
        * math.cos(lat2)
        * math.sin(dlon / 2) ** 2
    )

    c = 2 * math.atan2(
        math.sqrt(a),
        math.sqrt(1 - a)
    )

    return R * c


def format_time(minutes):
    minutes = int(round(minutes))

    hour = (minutes // 60) % 24
    minute = minutes % 60

    suffix = "AM" if hour < 12 else "PM"

    display_hour = hour % 12

    if display_hour == 0:
        display_hour = 12

    return f"{display_hour}:{minute:02d} {suffix}"


In [7]:
df["rating_score"] = minmax(df["rating"])

df["popularity_score"] = minmax(
    np.log1p(
        df["reviews"].clip(lower=0)
    )
)

df["hybrid_base_score"] = minmax(
    df["hybrid_score"]
)

df[[
    "name",
    "rating_score",
    "popularity_score",
    "hybrid_base_score"
]].head()


,name,rating_score,popularity_score,hybrid_base_score
0,Hadimba Devi Temple,0.777778,1.000000,0.895894
1,Old Manali snow point,0.777778,0.429428,0.868059
2,Nehru Kund,0.555556,0.777182,0.758313
3,Kullu Manali River rafting,0.666667,0.240583,0.322953
4,Jogini Falls,0.777778,0.817225,1.000000


In [8]:
def recommend_places(
    query,
    user_preferences,
    top_n=5
):
    missing = [
        feature
        for feature in FEATURE_COLUMNS
        if feature not in user_preferences
    ]

    if missing:
        raise ValueError(
            f"Missing preference features: {missing}"
        )

    # Structured similarity
    user_vector = np.array([
        float(user_preferences[feature])
        for feature in FEATURE_COLUMNS
    ]).reshape(1, -1)

    structured_scores = cosine_similarity(
        user_vector,
        X_STRUCTURED
    ).flatten()

    # TF-IDF
    query_vector = tfidf_vectorizer.transform(
        [query]
    )

    tfidf_scores = cosine_similarity(
        query_vector,
        tfidf_matrix
    ).flatten()

    # Semantic
    query_embedding = semantic_model.encode(
        [query],
        normalize_embeddings=True
    )

    semantic_scores = cosine_similarity(
        query_embedding,
        place_embeddings
    ).flatten()

    result = df.copy()

    result["structured_score"] = structured_scores
    result["tfidf_score"] = tfidf_scores
    result["semantic_score"] = semantic_scores

    result["api_score"] = (
        0.30 * result["structured_score"]
        + 0.25 * result["tfidf_score"]
        + 0.30 * result["semantic_score"]
        + 0.10 * result["rating_score"]
        + 0.05 * result["popularity_score"]
    )

    return (
        result
        .sort_values("api_score", ascending=False)
        .head(top_n)
        .reset_index(drop=True)
    )


In [9]:
test_preferences = {
    "nature": 1.0,
    "history": 0.0,
    "culture": 0.0,
    "adventure": 0.4,
    "photography": 1.0,
    "shopping": 0.0,
    "religious": 0.0,
    "family": 0.2
}

recommendations = recommend_places(
    "peaceful scenic places for taking beautiful photos",
    test_preferences,
    top_n=5
)

recommendations[[
    "name",
    "activity_type",
    "rating",
    "reviews",
    "semantic_score",
    "api_score"
]]


,name,activity_type,rating,reviews,semantic_score,api_score
0,Jogini Falls,waterfall,4.6,10842,0.490182,0.488179
1,Manali View Point,viewpoint,4.6,87,0.557698,0.459309
2,Van Vihar National Park,nature,4.2,9050,0.511589,0.449073
3,Hadimba Devi Temple,temple,4.6,49688,0.449838,0.427874
4,Old Manali snow point,winter_experience,4.6,428,0.425539,0.425908


In [10]:
AVERAGE_SPEED_KMPH = 25
DAY_START_MINUTES = 9 * 60
MAX_DAY_MINUTES = 7 * 60


def generate_itinerary(candidates, num_days=3):
    candidates = candidates.copy().reset_index(drop=True)

    if candidates.empty:
        return []

    # Build pairwise distances for this candidate set.
    n = len(candidates)
    distances = np.zeros((n, n))

    for i in range(n):
        for j in range(n):
            distances[i, j] = haversine_km(
                candidates.loc[i, "latitude"],
                candidates.loc[i, "longitude"],
                candidates.loc[j, "latitude"],
                candidates.loc[j, "longitude"]
            )

    travel_times = (
        distances / AVERAGE_SPEED_KMPH
    ) * 60

    # Greedy balanced allocation.
    remaining = set(range(n))
    itinerary = []

    for day in range(1, num_days + 1):
        if not remaining:
            break

        # Start from the best remaining candidate.
        anchor = max(
            remaining,
            key=lambda idx: candidates.loc[
                idx,
                "api_score"
            ]
        )

        current = anchor
        used_minutes = 0
        day_stops = []

        while remaining:
            feasible = []

            for idx in remaining:
                travel = (
                    0
                    if not day_stops
                    else travel_times[current, idx]
                )

                visit = float(
                    candidates.loc[
                        idx,
                        "estimated_visit_minutes"
                    ]
                )

                total = travel + visit

                if (
                    used_minutes + total
                    <= MAX_DAY_MINUTES
                ):
                    efficiency = (
                        candidates.loc[
                            idx,
                            "api_score"
                        ]
                        / (1 + travel)
                    )

                    feasible.append(
                        (
                            idx,
                            travel,
                            visit,
                            efficiency
                        )
                    )

            if not feasible:
                break

            idx, travel, visit, _ = max(
                feasible,
                key=lambda item: item[3]
            )

            arrival = (
                DAY_START_MINUTES
                + used_minutes
                + travel
            )

            departure = (
                arrival + visit
            )

            day_stops.append({
                "day": day,
                "stop": len(day_stops) + 1,
                "place": candidates.loc[
                    idx,
                    "name"
                ],
                "activity_type": candidates.loc[
                    idx,
                    "activity_type"
                ],
                "arrival": format_time(arrival),
                "departure": format_time(departure),
                "travel_before_minutes":
                    round(travel, 1),
                "visit_minutes":
                    int(visit),
                "estimated_price_level":
                    int(candidates.loc[
                        idx,
                        "estimated_price_level"
                    ]),
                "score":
                    round(
                        float(candidates.loc[
                            idx,
                            "api_score"
                        ]),
                        4
                    ),
                "latitude": float(
                    candidates.loc[
                        idx,
                        "latitude"
                    ]
                ),
                "longitude": float(
                    candidates.loc[
                        idx,
                        "longitude"
                    ]
                )
            })

            used_minutes += total
            current = idx
            remaining.remove(idx)

        itinerary.extend(day_stops)

    return itinerary


In [11]:
class PreferenceModel(BaseModel):
    nature: float = Field(0.0, ge=0.0, le=1.0)
    history: float = Field(0.0, ge=0.0, le=1.0)
    culture: float = Field(0.0, ge=0.0, le=1.0)
    adventure: float = Field(0.0, ge=0.0, le=1.0)
    photography: float = Field(0.0, ge=0.0, le=1.0)
    shopping: float = Field(0.0, ge=0.0, le=1.0)
    religious: float = Field(0.0, ge=0.0, le=1.0)
    family: float = Field(0.0, ge=0.0, le=1.0)


class RecommendationRequest(BaseModel):
    destination: str = "Manali"
    query: str
    days: int = Field(3, ge=1, le=14)
    top_n: int = Field(5, ge=1, le=15)
    preferences: PreferenceModel


In [12]:
app = FastAPI(
    title="TravelMate AI API",
    description="AI-powered travel recommendation and itinerary backend",
    version="1.0.0"
)


@app.get("/")
def root():
    return {
        "app": "TravelMate AI",
        "status": "running",
        "message": "Personalized travel AI backend"
    }


@app.get("/health")
def health():
    return {
        "status": "healthy",
        "places_loaded": len(df)
    }


@app.post("/recommend")
def recommend(request: RecommendationRequest):

    preferences = (
        request.preferences
        .model_dump()
    )

    results = recommend_places(
        query=request.query,
        user_preferences=preferences,
        top_n=request.top_n
    )

    # Only return JSON-safe fields.
    output = results[[
        "name",
        "activity_type",
        "travel_style",
        "rating",
        "reviews",
        "latitude",
        "longitude",
        "estimated_visit_minutes",
        "estimated_price_level",
        "semantic_score",
        "tfidf_score",
        "structured_score",
        "api_score"
    ]].copy()

    return {
        "destination": request.destination,
        "query": request.query,
        "recommendations": output.to_dict(
            orient="records"
        )
    }


@app.post("/itinerary")
def itinerary(request: RecommendationRequest):

    preferences = (
        request.preferences
        .model_dump()
    )

    candidates = recommend_places(
        query=request.query,
        user_preferences=preferences,
        top_n=min(
            max(request.top_n, request.days * 3),
            len(df)
        )
    )

    plan = generate_itinerary(
        candidates,
        num_days=request.days
    )

    return {
        "destination": request.destination,
        "days": request.days,
        "query": request.query,
        "itinerary": plan
    }

print("FastAPI application created! ✅")


FastAPI application created! ✅


In [13]:
test_request = RecommendationRequest(
    destination="Manali",
    query="peaceful scenic places for photography",
    days=3,
    top_n=6,
    preferences=PreferenceModel(
        nature=1.0,
        history=0.0,
        culture=0.0,
        adventure=0.3,
        photography=1.0,
        shopping=0.0,
        religious=0.0,
        family=0.2
    )
)

recommend_response = recommend(
    test_request
)

recommend_response


{'destination': 'Manali',
 'query': 'peaceful scenic places for photography',
 'recommendations': [{'name': 'Jogini Falls',
   'activity_type': 'waterfall',
   'travel_style': 'culture, family, nature, photography',
   'rating': 4.6,
   'reviews': 10842,
   'latitude': 32.2750756,
   'longitude': 77.1881463,
   'estimated_visit_minutes': 90,
   'estimated_price_level': 1,
   'semantic_score': 0.46041497588157654,
   'tfidf_score': 0.2389770647810099,
   'structured_score': 0.7537075808102849,
   'api_score': 0.5426200752592187},
  {'name': 'Manali View Point',
   'activity_type': 'viewpoint',
   'travel_style': 'photography',
   'rating': 4.6,
   'reviews': 87,
   'latitude': 32.2338352,
   'longitude': 77.1873592,
   'estimated_visit_minutes': 45,
   'estimated_price_level': 1,
   'semantic_score': 0.573796272277832,
   'tfidf_score': 0.26884415501509756,
   'structured_score': 0.6851887098275317,
   'api_score': 0.5346456429336003},
  {'name': 'Van Vihar National Park',
   'activity_

In [14]:
itinerary_response = itinerary(
    test_request
)

pd.DataFrame(
    itinerary_response["itinerary"]
)


,day,stop,place,activity_type,arrival,departure,travel_before_minutes,visit_minutes,estimated_price_level,score,latitude,longitude
0,1,1,Jogini Falls,waterfall,9:00 AM,10:30 AM,0.0,90,1,0.5426,32.275076,77.188146
1,1,2,Nehru Kund,sightseeing,10:33 AM,11:33 AM,3.5,60,1,0.4469,32.285982,77.179824
2,1,3,Old Manali snow point,winter_experience,12:28 PM,1:58 PM,9.8,90,1,0.4754,32.249112,77.180076
3,1,4,Hadimba Devi Temple,temple,2:10 PM,3:10 PM,0.4,60,2,0.4658,32.248353,77.181573
4,2,1,Manali View Point,viewpoint,9:00 AM,9:45 AM,0.0,45,1,0.5346,32.233835,77.187359
5,2,2,Van Vihar National Park,nature,10:31 AM,12:31 PM,1.5,120,1,0.4980,32.239113,77.189089
6,2,3,Lama Dugh Trek Start Point,trekking,12:11 PM,2:41 PM,4.1,150,1,0.4121,32.248903,77.175013
7,2,4,Kharma valley,nature,1:48 PM,3:48 PM,2.2,120,1,0.3836,32.254802,77.168016
8,3,1,Baror Parsha Waterfall,waterfall,9:00 AM,10:30 AM,0.0,90,1,0.3689,32.206785,77.184900


In [15]:
app_code = 'import math\nimport numpy as np\nimport pandas as pd\n\nfrom fastapi import FastAPI\nfrom pydantic import BaseModel, Field\nfrom sentence_transformers import SentenceTransformer\nfrom sklearn.feature_extraction.text import TfidfVectorizer\nfrom sklearn.metrics.pairwise import cosine_similarity\n\n\nDATA_PATH = "data/processed/manali_travel_enriched.csv"\nEMBEDDING_PATH = "data/embeddings/manali_place_embeddings.npy"\n\nFEATURE_COLUMNS = [\n    "nature",\n    "history",\n    "culture",\n    "adventure",\n    "photography",\n    "shopping",\n    "religious",\n    "family"\n]\n\nAVERAGE_SPEED_KMPH = 25\nDAY_START_MINUTES = 9 * 60\nMAX_DAY_MINUTES = 7 * 60\n\n\ndf = pd.read_csv(DATA_PATH)\nplace_embeddings = np.load(EMBEDDING_PATH)\n\nX_STRUCTURED = (\n    df[FEATURE_COLUMNS]\n    .fillna(0)\n    .astype(float)\n)\n\ndf["recommendation_text"] = (\n    df["name"].fillna("") + " "\n    + df["category"].fillna("") + " "\n    + df["travel_tags"].fillna("")\n)\n\ntfidf_vectorizer = TfidfVectorizer(\n    stop_words="english",\n    ngram_range=(1, 2)\n)\n\ntfidf_matrix = tfidf_vectorizer.fit_transform(\n    df["recommendation_text"].astype(str)\n)\n\nsemantic_model = SentenceTransformer(\n    "sentence-transformers/all-MiniLM-L6-v2"\n)\n\n\ndef minmax(series):\n    series = series.astype(float)\n\n    if series.max() == series.min():\n        return pd.Series(\n            np.ones(len(series)),\n            index=series.index\n        )\n\n    return (\n        (series - series.min())\n        / (series.max() - series.min())\n    )\n\n\ndf["rating_score"] = minmax(df["rating"])\ndf["popularity_score"] = minmax(\n    np.log1p(df["reviews"].clip(lower=0))\n)\n\n\ndef haversine_km(lat1, lon1, lat2, lon2):\n    R = 6371.0\n\n    lat1 = math.radians(lat1)\n    lon1 = math.radians(lon1)\n    lat2 = math.radians(lat2)\n    lon2 = math.radians(lon2)\n\n    dlat = lat2 - lat1\n    dlon = lon2 - lon1\n\n    a = (\n        math.sin(dlat / 2) ** 2\n        + math.cos(lat1)\n        * math.cos(lat2)\n        * math.sin(dlon / 2) ** 2\n    )\n\n    c = 2 * math.atan2(\n        math.sqrt(a),\n        math.sqrt(1 - a)\n    )\n\n    return R * c\n\n\ndef format_time(minutes):\n    minutes = int(round(minutes))\n\n    hour = (minutes // 60) % 24\n    minute = minutes % 60\n\n    suffix = "AM" if hour < 12 else "PM"\n    display_hour = hour % 12 or 12\n\n    return f"{display_hour}:{minute:02d} {suffix}"\n\n\ndef recommend_places(\n    query,\n    user_preferences,\n    top_n=5\n):\n    user_vector = np.array([\n        float(user_preferences[feature])\n        for feature in FEATURE_COLUMNS\n    ]).reshape(1, -1)\n\n    structured_scores = cosine_similarity(\n        user_vector,\n        X_STRUCTURED\n    ).flatten()\n\n    query_vector = tfidf_vectorizer.transform([query])\n\n    tfidf_scores = cosine_similarity(\n        query_vector,\n        tfidf_matrix\n    ).flatten()\n\n    query_embedding = semantic_model.encode(\n        [query],\n        normalize_embeddings=True\n    )\n\n    semantic_scores = cosine_similarity(\n        query_embedding,\n        place_embeddings\n    ).flatten()\n\n    result = df.copy()\n\n    result["structured_score"] = structured_scores\n    result["tfidf_score"] = tfidf_scores\n    result["semantic_score"] = semantic_scores\n\n    result["api_score"] = (\n        0.30 * result["structured_score"]\n        + 0.25 * result["tfidf_score"]\n        + 0.30 * result["semantic_score"]\n        + 0.10 * result["rating_score"]\n        + 0.05 * result["popularity_score"]\n    )\n\n    return (\n        result\n        .sort_values("api_score", ascending=False)\n        .head(top_n)\n        .reset_index(drop=True)\n    )\n\n\ndef generate_itinerary(candidates, num_days=3):\n    candidates = candidates.copy().reset_index(drop=True)\n\n    if candidates.empty:\n        return []\n\n    n = len(candidates)\n    distances = np.zeros((n, n))\n\n    for i in range(n):\n        for j in range(n):\n            distances[i, j] = haversine_km(\n                candidates.loc[i, "latitude"],\n                candidates.loc[i, "longitude"],\n                candidates.loc[j, "latitude"],\n                candidates.loc[j, "longitude"]\n            )\n\n    travel_times = (\n        distances / AVERAGE_SPEED_KMPH\n    ) * 60\n\n    remaining = set(range(n))\n    itinerary = []\n\n    for day in range(1, num_days + 1):\n        if not remaining:\n            break\n\n        current = max(\n            remaining,\n            key=lambda idx: candidates.loc[\n                idx, "api_score"\n            ]\n        )\n\n        used_minutes = 0\n        day_stops = []\n\n        while remaining:\n            feasible = []\n\n            for idx in remaining:\n                travel = (\n                    0\n                    if not day_stops\n                    else travel_times[current, idx]\n                )\n\n                visit = float(\n                    candidates.loc[\n                        idx,\n                        "estimated_visit_minutes"\n                    ]\n                )\n\n                total = travel + visit\n\n                if (\n                    used_minutes + total\n                    <= MAX_DAY_MINUTES\n                ):\n                    efficiency = (\n                        candidates.loc[\n                            idx,\n                            "api_score"\n                        ]\n                        / (1 + travel)\n                    )\n\n                    feasible.append(\n                        (idx, travel, visit, efficiency)\n                    )\n\n            if not feasible:\n                break\n\n            idx, travel, visit, _ = max(\n                feasible,\n                key=lambda item: item[3]\n            )\n\n            arrival = (\n                DAY_START_MINUTES\n                + used_minutes\n                + travel\n            )\n\n            departure = arrival + visit\n\n            day_stops.append({\n                "day": day,\n                "stop": len(day_stops) + 1,\n                "place": candidates.loc[idx, "name"],\n                "activity_type": candidates.loc[\n                    idx, "activity_type"\n                ],\n                "arrival": format_time(arrival),\n                "departure": format_time(departure),\n                "travel_before_minutes": round(travel, 1),\n                "visit_minutes": int(visit),\n                "estimated_price_level": int(\n                    candidates.loc[\n                        idx,\n                        "estimated_price_level"\n                    ]\n                ),\n                "score": round(\n                    float(\n                        candidates.loc[idx, "api_score"]\n                    ),\n                    4\n                ),\n                "latitude": float(\n                    candidates.loc[idx, "latitude"]\n                ),\n                "longitude": float(\n                    candidates.loc[idx, "longitude"]\n                )\n            })\n\n            used_minutes += total\n            current = idx\n            remaining.remove(idx)\n\n        itinerary.extend(day_stops)\n\n    return itinerary\n\n\nclass PreferenceModel(BaseModel):\n    nature: float = Field(0.0, ge=0.0, le=1.0)\n    history: float = Field(0.0, ge=0.0, le=1.0)\n    culture: float = Field(0.0, ge=0.0, le=1.0)\n    adventure: float = Field(0.0, ge=0.0, le=1.0)\n    photography: float = Field(0.0, ge=0.0, le=1.0)\n    shopping: float = Field(0.0, ge=0.0, le=1.0)\n    religious: float = Field(0.0, ge=0.0, le=1.0)\n    family: float = Field(0.0, ge=0.0, le=1.0)\n\n\nclass RecommendationRequest(BaseModel):\n    destination: str = "Manali"\n    query: str\n    days: int = Field(3, ge=1, le=14)\n    top_n: int = Field(5, ge=1, le=15)\n    preferences: PreferenceModel\n\n\napp = FastAPI(\n    title="TravelMate AI API",\n    description="AI-powered travel recommendation and itinerary backend",\n    version="1.0.0"\n)\n\n\n@app.get("/")\ndef root():\n    return {\n        "app": "TravelMate AI",\n        "status": "running",\n        "message": "Personalized travel AI backend"\n    }\n\n\n@app.get("/health")\ndef health():\n    return {\n        "status": "healthy",\n        "places_loaded": len(df)\n    }\n\n\n@app.post("/recommend")\ndef recommend(request: RecommendationRequest):\n    results = recommend_places(\n        query=request.query,\n        user_preferences=request.preferences.model_dump(),\n        top_n=request.top_n\n    )\n\n    output = results[[\n        "name",\n        "activity_type",\n        "travel_style",\n        "rating",\n        "reviews",\n        "latitude",\n        "longitude",\n        "estimated_visit_minutes",\n        "estimated_price_level",\n        "semantic_score",\n        "tfidf_score",\n        "structured_score",\n        "api_score"\n    ]].copy()\n\n    return {\n        "destination": request.destination,\n        "query": request.query,\n        "recommendations": output.to_dict(\n            orient="records"\n        )\n    }\n\n\n@app.post("/itinerary")\ndef itinerary(request: RecommendationRequest):\n    candidates = recommend_places(\n        query=request.query,\n        user_preferences=request.preferences.model_dump(),\n        top_n=min(\n            max(request.top_n, request.days * 3),\n            len(df)\n        )\n    )\n\n    plan = generate_itinerary(\n        candidates,\n        num_days=request.days\n    )\n\n    return {\n        "destination": request.destination,\n        "days": request.days,\n        "query": request.query,\n        "itinerary": plan\n    }\n'

app_path = Path("../app")
app_path.mkdir(parents=True, exist_ok=True)

main_path = app_path / "main.py"
main_path.write_text(
    app_code,
    encoding="utf-8"
)

print(f"✅ Backend created: {main_path}")


✅ Backend created: ..\app\main.py
